In [14]:
import numpy as np
import pandas as pd

In [15]:
data = pd.read_csv('vehicle_predictive_maintenance_dataset.csv')

In [16]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   mileage       100000 non-null  int64  
 1   engine_temp   100000 non-null  float64
 2   oil_pressure  100000 non-null  float64
 3   vibration     100000 non-null  float64
 4   brake_wear    100000 non-null  float64
 5   rpm_variance  100000 non-null  float64
 6   failure       100000 non-null  int64  
dtypes: float64(5), int64(2)
memory usage: 5.3 MB


In [17]:
data.head()

,mileage,engine_temp,oil_pressure,vibration,brake_wear,rpm_variance,failure
0,121958,75.486846,36.720794,0.186105,81.364904,43.537536,0
1,146867,101.481892,35.996380,0.394017,13.946840,117.360958,0
2,131932,82.751808,40.553054,0.095191,41.904289,164.699605,0
3,103694,88.601436,42.580856,0.462193,21.820569,122.341751,0
4,119879,95.187651,36.522665,0.180342,2.597110,126.580960,0


In [18]:
data.describe()

,mileage,engine_temp,oil_pressure,vibration,brake_wear,rpm_variance,failure
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,99612.225580,90.025526,40.020289,0.252798,50.071617,122.197169,0.105040
std,57797.665634,7.993050,9.979742,0.143836,28.800564,75.415525,0.306606
min,2.000000,60.000000,10.000000,0.000000,0.002713,0.000000,0.000000
25%,49556.000000,84.626282,33.285473,0.148599,25.274407,65.849420,0.000000
50%,99482.000000,89.982199,40.035583,0.250111,50.007371,119.955046,0.000000
75%,149443.250000,95.402068,46.743151,0.350938,74.990438,174.031231,0.000000
max,199994.000000,120.000000,88.015146,0.975268,99.999691,489.529415,1.000000


In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix,roc_auc_score)
import joblib

X = data.drop(['failure'], axis=1)
y = data['failure']
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

# Model Pipeline (UNCHANGED CORE MODEL)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8
    ))
])

print("Training model...")
model.fit(X_train, y_train)

print("Model Training Completed...")

Training model...
Model Training Completed...


In [24]:
print("Evaluating model performance...")

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy      : {accuracy*100:.4f} %")


Evaluating model performance...
Accuracy      : 89.4850 %


In [25]:
roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score : {roc_auc:.4f}")

ROC-AUC Score : 0.5462


In [26]:
print("Confusion Matrix")
cm = confusion_matrix(y_test, y_pred)
print(cm)

Confusion Matrix
[[17896     3]
 [ 2100     1]]


In [27]:
print("Classification Report")
print(classification_report(y_test, y_pred, digits=4))

Classification Report
              precision    recall  f1-score   support

           0     0.8950    0.9998    0.9445     17899
           1     0.2500    0.0005    0.0010      2101

    accuracy                         0.8949     20000
   macro avg     0.5725    0.5002    0.4727     20000
weighted avg     0.8272    0.8949    0.8454     20000



In [28]:
joblib.dump(model, "vehicle_model.pkl")
print("Model trained and saved as vehicle_model.pkl")

Model trained and saved as vehicle_model.pkl
